<a href="https://colab.research.google.com/github/pillaiganesh/MyAIAdventures/blob/HAAI%2B%2B/HAAI%2B%2B_Theory_Model_Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Quantization - Lecture Outline

## I. Motivation & Problem Statement
- **LLM Deployment Challenges**
  - LLaMA-2 7B: 14GB GPU memory for inference alone
  - Larger models like GPT-3 175B: >350GB requirements
  - Real barriers: Most consumer GPUs have 8-24GB VRAM
  - Production costs: GPU memory is expensive at scale
- **Two Critical Scenarios**
  - **Inference deployment**: Serving models efficiently on limited hardware
  - **Training/fine-tuning**: Adapting models when full retraining is prohibitive
  - **The gap**: Model sizes growing faster than hardware capabilities



## II. Quantization Fundamentals

### Core Concept
- **Definition**: Reducing numerical precision while preserving model functionality
- **Common transformations**: FP32 → FP16 (2x savings), FP32 → INT8 (4x savings)
- **Key insight**: Neural networks are surprisingly robust to precision reduction
- **Why it works**: Weights often contain redundant precision that doesn't affect final predictions

### Mathematical Foundation
- **Linear Quantization**: $Q(r) = \text{round}\left(\frac{r}{S}\right) + Z$
- **Key Parameters**: Scale factor ($S$), Zero point ($Z$)
- **Dequantization**: $\tilde{r} = S \cdot (Q(r) - Z)$

### Symmetric vs Asymmetric
- **Symmetric**: $Z = 0$, range centered around zero
  - Simpler computation, hardware-friendly
  - Best for weight distributions centered near zero
- **Asymmetric**: Uses full available quantization range
  - Better utilization when data is skewed (e.g., ReLU activations)
  - Extra computational cost due to zero-point handling



## III. Common Quantization Formats

### Half Precision (FP16)
- **Structure**: 1 sign + 5 exponent + 10 mantissa bits
- **Memory reduction**: Exactly 50%, widely supported by modern hardware
- **Trade-offs**: Minimal accuracy loss, but reduced dynamic range can cause training instability
- **Best use**: Inference and stable training scenarios

### Brain Float (BF16)  
- **Structure**: 1 sign + 8 exponent + 7 mantissa bits
- **Key advantage**: Same exponent range as FP32 (better numerical stability)
- **Google's innovation**: Designed specifically for machine learning workloads
- **Trade-off**: Less mantissa precision than FP16, but more robust gradients

### Integer Quantization
- **INT8**: 256 discrete levels, 75% memory reduction from FP32
  - Requires calibration dataset to determine optimal mapping
  - Well-supported by inference engines (TensorRT, ONNX Runtime)
- **INT4**: 16 discrete levels, 87.5% memory reduction
  - Aggressive quantization, needs sophisticated techniques to maintain quality
  - Enables deployment of 7B models on consumer hardware



## IV. Memory Consumption Analysis

### Training Requirements
- **Memory components breakdown**:
  - **Model weights**: Base parameter storage (e.g., 28GB for LLaMA-2 7B in FP32)
  - **Gradients**: Same size as weights (another 28GB)
  - **Optimizer states**: Adam needs momentum + variance (56GB more)
  - **Activations**: Varies with batch size and sequence length
- **Total formula**: $\text{Memory} \approx 16P + \text{Activations}$ for FP32 training
- **Reality check**: 7B model needs ~112GB just for optimization, before activations!

### Inference Requirements  
- **Much simpler**: Only need weights + activations for forward pass
- **No gradients or optimizer states**: Massive memory savings opportunity
- **Formula**: $\text{Memory} \approx 4P + \text{Activations}$ (FP32)
- **Implication**: Quantization has bigger relative impact on inference than training



## V. Quantization Strategies

### Post-Training Quantization (PTQ)
- **Process**: Take trained model → convert to lower precision without retraining
- **Advantages**: Fast, no additional training compute required
- **Steps**: Weight conversion → activation calibration using sample data → layer sensitivity analysis
- **Limitation**: Aggressive quantization (INT4) often degrades accuracy significantly
- **Best for**: INT8 quantization where accuracy preservation is critical

### Quantization-Aware Training (QAT)
- **Process**: Simulate quantization during training so model adapts to constraints
- **Key technique**: Straight-through estimator (quantize forward, full-precision backward)
- **Advantage**: Better accuracy retention, especially for aggressive quantization
- **Cost**: Requires full retraining or significant fine-tuning compute
- **Best for**: When you have training compute and need maximum accuracy

### Static vs Dynamic
- **Static**: Determine all quantization parameters offline using calibration data
  - Predictable performance, optimal for deployment
  - Risk: Calibration data may not represent all inference scenarios
- **Dynamic**: Compute activation quantization parameters at runtime
  - Better accuracy but computational overhead during inference
  - Common pattern: Static weights + dynamic activations



## VI. Advanced Techniques: LoRA & QLoRA

### Low-Rank Adaptation (LoRA)
- **The problem**: Fine-tuning 7B+ models requires storing full gradients and optimizer states
- **LoRA insight**: Most adaptation happens in low-dimensional subspace
- **Method**: Freeze base weights $W_0$, learn low-rank update $\Delta W = BA$
- **Forward pass**: $h = W_0 x + BAx$ where $A \in \mathbb{R}^{d \times r}$, $B \in \mathbb{R}^{r \times k}$
- **Parameter reduction**: From $dk$ to $r(d+k)$ trainable parameters
- **Typical values**: $r = 8, 16, 64$ gives 99%+ reduction in trainable parameters

### QLoRA Components

#### 4-bit NormalFloat (NF4)
- **Key insight**: Neural network weights after training follow roughly normal distribution
- **Innovation**: Non-uniform quantization levels optimized for this distribution
- **Result**: 16 carefully chosen values that minimize expected quantization error
- **Advantage**: Better than uniform INT4 for typical weight distributions

#### Double Quantization
- **The issue**: Even quantization constants (scales, zero-points) consume memory
- **Solution**: Quantize the quantization parameters themselves
- **Implementation**: Store scale factors in FP16, then further quantize to 8-bit
- **Memory impact**: Additional ~0.1GB savings for large models

#### Paged Optimizers
- **Problem**: Optimizer states don't fit in GPU memory during fine-tuning
- **Solution**: Transparent paging between GPU and CPU memory
- **Mechanism**: Automatically move optimizer states to CPU when not actively used
- **User experience**: Prevents OOM errors without manual memory management
- **Performance**: Minimal impact since optimizer updates are less frequent than forward/backward



## VII. Practical Memory Savings

### LLaMA-2 7B Memory Analysis
- **FP32 baseline**: 7B × 4 bytes = 28GB (reference point)
- **FP16**: 14GB (enables deployment on high-end consumer GPUs like RTX 4090)
- **INT8**: 7GB (fits comfortably on mid-range GPUs like RTX 3080)
- **INT4**: 3.5GB (enables deployment on entry-level GPUs like RTX 3060)
- **Reality check**: These are weights-only; add ~2-4GB for activations depending on context length

### LoRA Parameter Efficiency Example
- **Original model**: 7B parameters need gradients + optimizer states = ~112GB training memory
- **LoRA approach**: Only ~16.8M trainable parameters (rank 64)
- **Training memory**: ~0.3GB for LoRA parameters vs 112GB for full fine-tuning
- **Key insight**: >99% reduction enables fine-tuning on single consumer GPU



## VIII. Implementation Guidelines

### Mixed Precision Strategies
- Critical layers (embeddings, output): Higher precision
- Transformer blocks: Quantized representations
- Layer-wise sensitivity consideration

### Deployment Decision Matrix
- **Inference-only**: PTQ with INT8/FP16
- **Fine-tuning needed**: QLoRA approach
- **Extreme constraints**: INT4 with careful calibration

### Hardware Considerations
- Consumer GPUs: INT4/QLoRA enables deployment
- Production systems: Balance memory, speed, accuracy
- Calibration dataset quality critical for accuracy